# Exploration de la source BCEAO

**Projet :** Plateforme d'automatisation de la prospection (veille AO / RFP / CFP)  
**Module :** 1 - Collecte  
**Source étudiée :** BCEAO (Banque Centrale des États de l'Afrique de l'Ouest)

## Objectif de ce carnet

Avant d'écrire le collecteur définitif et de concevoir la base de données, je veux comprendre
comment une source réelle expose ses appels d'offres : est-ce qu'on peut y accéder simplement,
quels champs sont disponibles, et sous quelle forme. Je documente ici toute ma démarche, y compris
les essais qui n'ont pas marché, pour garder une trace de pourquoi j'ai choisi cette source.

## 1. Comment j'en suis arrivée à la BCEAO

Je suis partie de la liste de sources fournie par l'entreprise (le fichier JSON). Avant de coder,
j'ai testé les URLs une par une pour voir lesquelles étaient vraiment exploitables. Ce que j'ai
constaté :

- **Les banques et assurances privées** (Attijari, UIB, ATB, Maghrebia...) : pas de page publique
  d'appels d'offres. Soit elles passent par une plateforme fournisseurs sur inscription (ex. Attijari
  Sourcing), soit elles ne publient rien en ligne. Donc non collectables sans compte fournisseur.
- **TUNEPS** (marchés publics tunisiens) : la consultation existe mais l'accès demande un certificat
  électronique TUNTRUST. Trop compliqué pour un premier essai.
- **L'ANPR** : beaucoup de bruit (la page mélange appels d'offres, bourses, recrutements de
  chercheurs). En plus l'URL de la liste pointait vers un mauvais domaine.
- **Plusieurs URLs de la liste renvoyaient une erreur 404** : les liens étaient périmés.

En testant, je suis tombée sur la **BCEAO**, qui elle répondait correctement (HTTP 200). En ouvrant
sa page d'appels d'offres, j'ai vu une vraie liste structurée, publique, avec des marchés récents.
C'est donc une bonne candidate comme **source pilote** : si j'arrive à faire fonctionner toute la
chaîne dessus, je pourrai ensuite l'étendre aux autres sources.

Remarque : la BCEAO est la banque centrale de l'Afrique de l'Ouest (UEMOA).

## 2. Récupération de la page

Je commence par récupérer le HTML de la page des appels d'offres. J'ajoute un `User-Agent` de
navigateur, parce que beaucoup de sites bloquent les requêtes qui n'en ont pas (elles ressemblent
à des robots).

In [34]:
import requests
import re
import json
from bs4 import BeautifulSoup

URL = "https://www.bceao.int/fr/appels-offres/appels-offres-marches-publics-achats"
BASE = "https://www.bceao.int"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/124.0 Safari/537.36"}

reponse = requests.get(URL, headers=HEADERS, timeout=30)
print("Code HTTP :", reponse.status_code)
print("Taille de la page :", len(reponse.content), "octets")

Code HTTP : 200
Taille de la page : 132597 octets


## 3. Première observation importante

Quand j'ai exploré l'ONMP (un autre portail), le tableau des annonces était vide dans le HTML :
le contenu était chargé en JavaScript après l'ouverture de la page. Dans ce cas il aurait fallu
un navigateur automatisé (Playwright), ce qui est plus lourd.

Pour la BCEAO, je vérifie si les annonces sont **directement dans le HTML**. Si oui, un simple
`requests` suffit, pas besoin de navigateur. Je cherche les liens qui pointent vers une page
d'annonce (elles ont toutes une URL du type `/fr/appels-offres/<titre>`).

## Vérification : le contenu est-il dans le HTML brut ?

Avant d'extraire quoi que ce soit, je veux savoir si je peux me contenter de `requests` ou s'il
me faudra un navigateur automatisé. Le test : est-ce que les annonces sont déjà présentes dans le
HTML que le serveur m'a envoyé, ou est-ce qu'elles sont chargées après coup en JavaScript ?

- Si je **trouve** des liens d'annonces dans le HTML brut → le contenu est déjà là (rendu côté
  serveur) → `requests` suffit.
- Si je n'en **trouve aucun** alors que le site en affiche → le contenu est chargé en JavaScript
  (rendu côté client) → il faudrait Playwright.

C'est exactement sur ce point que la BCEAO et l'ONMP diffèrent : sur l'ONMP, le tableau était vide
dans le HTML brut (cas JavaScript), alors qu'ici je veux vérifier ce qu'il en est pour la BCEAO.

In [35]:
# Je compte les liens d'annonces présents dans le HTML BRUT (sans exécuter de JavaScript).
# 'reponse.text' est exactement ce que le serveur a renvoyé, rien de plus.
liens_annonces = re.findall(r'/fr/appels-offres/[a-z0-9\-]{15,}', reponse.text)
nb = len(set(liens_annonces))

print(f"Liens d'annonces trouvés dans le HTML brut : {nb}")
print()

if nb > 0:
    print("=> CAS 1 : le contenu est déjà dans le HTML (rendu côté serveur).")
    print("   requests suffit, pas besoin de navigateur headless.")
else:
    print("=> CAS 2 : aucune annonce dans le HTML brut.")
    print("   Le contenu est probablement chargé en JavaScript : il faudrait Playwright.")

Liens d'annonces trouvés dans le HTML brut : 35

=> CAS 1 : le contenu est déjà dans le HTML (rendu côté serveur).
   requests suffit, pas besoin de navigateur headless.


In [36]:
soup = BeautifulSoup(reponse.text, "html.parser")

# Tous les liens vers une fiche d'annonce
liens = soup.find_all("a", href=re.compile(r"/fr/appels-offres/[a-z]"))

# On ne garde que ceux dont le texte ressemble à une annonce (il contient 'publié le')
annonces = [a for a in liens if "ublié le" in a.get_text()]

print("Nombre d'annonces trouvées dans le HTML :", len(annonces))
print("\nExemple de texte brut d'une annonce :")
print(" ".join(annonces[0].get_text().split()))

Nombre d'annonces trouvées dans le HTML : 32

Exemple de texte brut d'une annonce :
Publié le 16 juin 2026 Date limite le 30 Juillet 2026 Etudes d’ingénierie et construction de l’ensemble immobilier de l’Agence Auxiliaire de Kandi au Bénin


In [37]:
# Les liens attrapés par la regex mais qui ne sont PAS des annonces datées
tous = set(re.findall(r'/fr/appels-offres/[a-z0-9\-]{15,}', reponse.text))
vraies = set(a.get("href").split("?")[0].rsplit("/", 0)[0] for a in annonces)

# affiche simplement tous les liens uniques trouvés par la regex
for lien in sorted(tous):
    print(lien)

/fr/appels-offres/acquisition-dun-gerbeur-electrique-grande-levee-conducteur-accompagnant-et-son
/fr/appels-offres/ameublement-du-secretariat-general-de-la-commission-bancaire-de-lumoa-sgcb-umoa
/fr/appels-offres/appel-candidatures-pour-la-49e-promotion-du-cycle-diplomant-du-cofeb
/fr/appels-offres/appel-concurrence-pour-la-peinture-des-facades-des-immeubles-de-lagence-auxiliaire-de
/fr/appels-offres/appels-offres-marches-publics-achats
/fr/appels-offres/appels-offres-marches-titres-publics-prives
/fr/appels-offres/avis-dappel-doffres-pour-la-selection-dun-prestataire-pour-les-tests-dintrusion-de-la
/fr/appels-offres/etudes-dingenierie-en-genie-civil-et-la-construction-du-complexe-immobilier-de-la
/fr/appels-offres/etudes-dingenierie-et-construction-de-lensemble-immobilier-de-lagence-auxiliaire-de
/fr/appels-offres/fourniture-de-dix10-ventilos-convecteurs-et-de-vingt-20-cassettes-eau-glacee-lagence
/fr/appels-offres/fourniture-de-trois-03-serveurs-physiques-x86-et-dune-licence-manageen

## Vérification : quels liens ne sont PAS de vraies annonces ?

La recherche par expression régulière (sur le HTML brut) trouve un peu plus de liens que
l'extraction avec BeautifulSoup. La différence vient de liens qui ont la **forme** d'une URL
d'annonce (`/fr/appels-offres/...`) mais qui ne sont pas des appels d'offres datés : le lien
vers la page elle-même, des appels à candidatures, ou des opérations de marché monétaire.

Pour les repérer, j'affiche les liens de cette forme dont le texte **ne contient pas** « Publié le ».
Ce sont eux qui expliquent l'écart, et c'est précisément ce que mon filtre `"ublié le"` élimine.

In [38]:
# Liens ayant la forme d'une annonce mais SANS "Publié le" : ceux qui font la différence
liens = soup.find_all("a", href=re.compile(r"/fr/appels-offres/[a-z]"))
non_annonces = [a for a in liens if "ublié le" not in a.get_text()]

# on évite d'afficher deux fois le même lien
vus = set()
print("Liens écartés (pas de vraies annonces) :\n")
for a in non_annonces:
    href = a.get("href")
    if href not in vus:
        vus.add(href)
        texte = " ".join(a.get_text().split())
        print("URL  :", href)
        print("Texte:", repr(texte[:70]))
        print("-" * 60)

Liens écartés (pas de vraies annonces) :

URL  : /fr/appels-offres/appels-offres-marches-publics-achats
Texte: 'FR'
------------------------------------------------------------
URL  : /fr/appels-offres/appel-candidatures-pour-la-49e-promotion-du-cycle-diplomant-du-cofeb
Texte: 'COFEB'
------------------------------------------------------------
URL  : /fr/appels-offres/marche-mon%C3%A9taire
Texte: 'Appel d’offres du marché'
------------------------------------------------------------
URL  : /fr/appels-offres/appels-offres-marches-titres-publics-prives
Texte: 'Avis d’appel d’offres'
------------------------------------------------------------


Les annonces sont bien présentes dans le HTML (j'en trouve une trentaine). **Donc pas besoin de
navigateur headless pour la BCEAO**, ce qui simplifie beaucoup la collecte.

En regardant le texte brut, je remarque qu'une annonce suit toujours le même modèle :

```
Publié le <date de publication> <référence éventuelle> Date limite le <date limite> <titre>
```

Je vais donc écrire une expression régulière pour découper ce texte en champs séparés.

In [39]:
import re

def trouver_section(soup, mot_cle):
    """Renvoie le <div> qui suit le <h2> contenant mot_cle (ex: 'en cours')."""
    for h2 in soup.find_all("h2"):
        if mot_cle in " ".join(h2.get_text().split()).lower():
            return h2.find_next_sibling()
    return None

def extraire_section(soup, mot_cle, statut_source):
    section = trouver_section(soup, mot_cle)
    if section is None:
        return []
    liens = section.find_all("a", href=re.compile(r"/fr/appels-offres/[a-z]"))
    annonces = [a for a in liens if "ublié le" in a.get_text()]
    # ... ici on applique ton extraction habituelle (le motif regex sur le texte) ...
    # et on ajoute à chaque annonce : "statut_source": statut_source
    return annonces

# usage
en_cours = extraire_section(soup, "en cours", "en cours")
closes   = extraire_section(soup, "clos", "clôturé")

## Inspection de la structure HTML

Avant d'écrire l'extraction, je dois comprendre comment une annonce est construite dans le HTML :
quelles balises l'entourent, où sont le titre, les dates, le lien. Je ne peux pas deviner le motif
d'extraction sans avoir regardé le code de la page.

Je prends donc UNE annonce et j'affiche son HTML brut pour voir les balises. C'est cette inspection
qui me dira sur quoi m'appuyer pour extraire les champs.

In [40]:
# 1. Je regarde le HTML brut d'une seule annonce, pour voir comment elle est structurée.
premiere = annonces[0]
print("HTML brut de la première annonce :")
print(premiere.prettify()[:600])
print("\n" + "="*70)

# 2. Quelles informations sont disponibles sur cette balise <a> ?
print("Nom de la balise :", premiere.name)
print("Attributs (dont le lien href) :", premiere.attrs)
print("\nTexte visible :")
print(repr(premiere.get_text()))

HTML brut de la première annonce :
<a href="https://www.bceao.int/fr/appels-offres/etudes-dingenierie-et-construction-de-lensemble-immobilier-de-lagence-auxiliaire-de">
 <span class="infoFile">
  Publié le
  <time datetime="00Z">
   16 juin 2026
  </time>
 </span>
 <span class="descFile">
  <span class="subTtr">
   Date limite le
   <time datetime="00Z">
    30 Juillet 2026
   </time>
  </span>
  <span class="ttr">
   Etudes d’ingénierie et construction de l’ensemble immobilier de l’Agence Auxiliaire de Kandi au Bénin
  </span>
 </span>
 <span class="clear">
 </span>
</a>


Nom de la balise : a
Attributs (dont le lien href) : {'href': 'https://www.bceao.int/fr/appels-offres/etudes-dingenierie-et-construction-de-lensemble-immobilier-de-lagence-auxiliaire-de'}

Texte visible :
'\nPublié le 16 juin 2026\n\n  Date limite le 30 Juillet 2026\n Etudes d’ingénierie et construction de l’ensemble immobilier de l’Agence Auxiliaire de Kandi au Bénin \n\n'


## 4. Extraction des champs

Je récupère pour chaque annonce : la date de publication, la référence (quand elle existe),
la date limite, le titre, et le lien. Pour l'identifiant unique (qui me servira plus tard à
éviter les doublons), j'utilise la fin de l'URL (le *slug*), parce que je remarque que la
référence, elle, n'est pas toujours présente.

In [41]:
motif = re.compile(
    r"Publié le\s+(?P<pub>\d{1,2}\s+\w+\s+\d{4})\s+"
    r"(?P<ref>[A-Z0-9/\-°N ]+?)?\s*"
    r"Date limite le\s+(?P<lim>\d{1,2}\s+\w+\s+\d{4})\s+"
    r"(?P<titre>.+)",
    re.IGNORECASE,
)

resultats = []
for a in annonces:
    texte = " ".join(a.get_text().split())
    m = motif.search(texte)
    if not m:
        continue  # si le format ne correspond pas, je passe

    href = a.get("href")
    lien = href if href.startswith("http") else BASE + href

    reference = (m.group("ref") or "").strip()
    # je nettoie les références parasites (trop courtes ou juste un 'N')
    if reference and (len(reference) < 4 or reference.lower() == "n"):
        reference = ""

    resultats.append({
        "cle_unique": lien.rstrip("/").split("/")[-1],  # le slug de l'URL
        "reference": reference,
        "intitule": m.group("titre").strip(),
        "date_publication": m.group("pub"),
        "delai_soumission": m.group("lim"),
        "lien": lien,
        "source": "BCEAO",
    })

print("Annonces extraites proprement :", len(resultats))
print()
print(json.dumps(resultats[1], ensure_ascii=False, indent=2))

Annonces extraites proprement : 32

{
  "cle_unique": "selection-dun-prestataire-charge-de-realiser-la-refonte-de-lintranet-de-la-bceao",
  "reference": "DP/Z00/DBA/092/2026",
  "intitule": "Report - Sélection d'un prestataire chargé de réaliser la refonte de l'intranet de la BCEAO",
  "date_publication": "22 mai 2026",
  "delai_soumission": "02 Juillet 2026",
  "lien": "https://www.bceao.int/fr/appels-offres/selection-dun-prestataire-charge-de-realiser-la-refonte-de-lintranet-de-la-bceao",
  "source": "BCEAO"
}


## Contrôle de qualité des champs extraits

Avant de continuer, je vérifie sur les données réelles que mon extraction est fiable. Je veux
savoir, pour chaque champ : combien de valeurs sont vides ou manquantes ? C'est ce contrôle qui
justifie mes choix (par exemple, utiliser le slug plutôt que la référence comme clé unique).
Je vérifie aussi que la clé unique ne contient pas de doublons, sinon elle ne pourrait pas servir
d'identifiant.

In [42]:
total = len(resultats)
print(f"Nombre total d'annonces extraites : {total}\n")

# 1. Pour chaque champ, combien de valeurs vides ou manquantes ?
champs = ["cle_unique", "reference", "intitule", "date_publication", "delai_soumission", "lien"]
print("Champs vides ou manquants par colonne :")
for champ in champs:
    vides = sum(1 for x in resultats if not x.get(champ))
    print(f"  {champ:18} : {vides} vide(s) sur {total}")

# 2. La clé unique est-elle vraiment unique ? (pas de doublons)
cles = [x["cle_unique"] for x in resultats]
nb_doublons = len(cles) - len(set(cles))
print(f"\nDoublons sur la clé unique : {nb_doublons}")

# 3. Les dates ont-elles bien été reconnues au format attendu ?
import re
format_date = re.compile(r"\d{1,2}\s+\w+\s+\d{4}")
dates_ok = sum(1 for x in resultats if format_date.match(x["date_publication"] or ""))
print(f"Dates de publication au format reconnu : {dates_ok} / {total}")

Nombre total d'annonces extraites : 32

Champs vides ou manquants par colonne :
  cle_unique         : 0 vide(s) sur 32
  reference          : 15 vide(s) sur 32
  intitule           : 0 vide(s) sur 32
  date_publication   : 0 vide(s) sur 32
  delai_soumission   : 0 vide(s) sur 32
  lien               : 0 vide(s) sur 32

Doublons sur la clé unique : 0
Dates de publication au format reconnu : 32 / 32


## 5. Ce que je remarque sur les données (utile pour la base)

En regardant les résultats, je note deux choses importantes pour la suite :

1. **La référence est parfois vide.** Certaines annonces n'ont pas de numéro dans leur texte.
   Donc je ne peux pas l'utiliser comme clé unique. Le *slug* de l'URL, lui, est toujours présent
   et stable : c'est lui qui servira de `cle_unique` pour la déduplication.

2. **Les dates sont en français littéral** (ex. `16 juin 2026`, `30 Juillet 2026`). Sous cette
   forme je ne peux pas les comparer ni calculer de délais. Il faut les convertir au format
   standard `AAAA-MM-JJ` avant de les stocker. C'est l'étape de normalisation ci-dessous.

In [43]:
MOIS = {
    "janvier": 1, "février": 2, "fevrier": 2, "mars": 3, "avril": 4, "mai": 5,
    "juin": 6, "juillet": 7, "août": 8, "aout": 8, "septembre": 9,
    "octobre": 10, "novembre": 11, "décembre": 12, "decembre": 12,
}

def normaliser_date(txt):
    """Convertit une date française ('16 juin 2026') en format ISO ('2026-06-16')."""
    m = re.match(r"(\d{1,2})\s+(\w+)\s+(\d{4})", txt.strip().lower())
    if not m:
        return None
    jour, mois, annee = m.groups()
    num_mois = MOIS.get(mois)
    if not num_mois:
        return None
    return f"{annee}-{num_mois:02d}-{int(jour):02d}"

# J'applique la normalisation à toutes les annonces
for x in resultats:
    x["date_publication"] = normaliser_date(x["date_publication"])
    x["delai_soumission"] = normaliser_date(x["delai_soumission"])

print("Exemple après normalisation des dates :")
print(json.dumps(resultats[1], ensure_ascii=False, indent=2))

Exemple après normalisation des dates :
{
  "cle_unique": "selection-dun-prestataire-charge-de-realiser-la-refonte-de-lintranet-de-la-bceao",
  "reference": "DP/Z00/DBA/092/2026",
  "intitule": "Report - Sélection d'un prestataire chargé de réaliser la refonte de l'intranet de la BCEAO",
  "date_publication": "2026-05-22",
  "delai_soumission": "2026-07-02",
  "lien": "https://www.bceao.int/fr/appels-offres/selection-dun-prestataire-charge-de-realiser-la-refonte-de-lintranet-de-la-bceao",
  "source": "BCEAO"
}


## 6. Vérification : y a-t-il des annonces pertinentes pour PWN & PATCH ?

La BCEAO publie surtout des marchés de travaux et de fournitures (bâtiment, climatisation...).
Mais pas seulement. Je fais un filtre très grossier par mots-clés pour vérifier qu'il y a bien
des marchés liés à l'informatique / la sécurité, donc des opportunités potentielles pour
l'entreprise.

Attention : ce filtre par mots-clés n'est qu'une vérification rapide. Le vrai tri de pertinence
sera fait plus tard par le module de scoring (module 2).

In [44]:
mots_cles = ("informati", "numérique", "logiciel", "serveur", "site internet",
             "intranet", "cyber", "réseau", "fibre", "système d'information")

pertinentes = [x for x in resultats
               if any(mot in x["intitule"].lower() for mot in mots_cles)]

print(f"{len(pertinentes)} annonces à connotation informatique :\n")
for x in pertinentes:
    print(" -", x["intitule"])

3 annonces à connotation informatique :

 - Report - Sélection d'un prestataire chargé de réaliser la refonte de l'intranet de la BCEAO
 - Reprise du câblage fibre optique des immeubles fonctionnels du Siège
 - Fourniture de trois (03) serveurs physiques X86 et d'une licence ManageEngine Applications Manager version entreprise pour la BCEAO - Report


## 7. Sauvegarde du résultat

Je sauvegarde les annonces extraites dans un fichier JSON. Ce sera la matière première à insérer
dans la base de données à l'étape suivante.

In [45]:
with open("bceao_annonces.json", "w", encoding="utf-8") as f:
    json.dump(resultats, f, ensure_ascii=False, indent=2)

print("Sauvegardé :", len(resultats), "annonces dans bceao_annonces.json")

Sauvegardé : 32 annonces dans bceao_annonces.json


## 8. Conclusion et prochaines étapes

**Ce que cette exploration m'a appris :**

- La BCEAO est une source publique, accessible avec un simple `requests` (pas besoin de navigateur
  headless, contrairement à l'ONMP).
- Les annonces se découpent en champs nets : numéro AO, acheteur, objet, date limite, date de
  publication, lien.
- Deux points de nettoyage à prévoir : la référence parfois absente (donc clé = slug de l'URL),
  et les dates françaises à convertir en format ISO.
- La source contient bien des marchés informatiques, donc des opportunités réelles pour l'entreprise.

**Champs retenus pour la table `opportunites` :**
`cle_unique`, `reference`, `intitule`, `date_publication`, `delai_soumission`, `lien`, `source`.

**Prochaines étapes :**

1. Concevoir la table `opportunites` (+ la table `historique_statuts`) à partir de ces champs.
2. Créer la base PostgreSQL.
3. Brancher cet extracteur sur la base : insérer chaque annonce en évitant les doublons grâce à
   `cle_unique`.
4. Gérer la pagination de la BCEAO (la liste fait plusieurs pages, paramètre `?page=`).
5. Plus tard : ajouter le scoring (module 2) puis l'orchestration dans n8n.

**Limite connue :** je n'ai traité ici que la première page. La BCEAO en a beaucoup d'autres
(`?page=1`, `?page=2`...). Il faudra boucler sur les pages, mais sans tout récupérer à chaque fois :
une fois la déduplication en place, je ne garderai que les nouvelles annonces.